In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
import matplotlib.pyplot as plt

# --- Part 1: Data Loading and Preprocessing ---
print("--- Step 1: Loading and Preprocessing Data ---")

# Load the dataset
try:
    # Using the column names from your final progress report for consistency
    df = pd.read_csv('../../../data/data.csv')
    df.rename(columns={
        'Code': 'replicate_id',
        'Date of Introduction': 'start_date',
        'Date': 'observation_date',
        'count': 'cumulative_mite_count'
    }, inplace=True)
    print("Dataset loaded and columns renamed successfully.")
except FileNotFoundError:
    print("Error: 'data.csv' not found. Please ensure the dataset file is in the same directory.")
    exit()

# Convert date columns to datetime objects
df['observation_date'] = pd.to_datetime(df['observation_date'])
df['date_of_introduction'] = pd.to_datetime(df['date_of_introduction'])

# Set the observation date as the index for time-series operations
df.set_index('observation_date', inplace=True)

# --- Handle missing dates by resampling each replicate to a daily frequency ---
# This is a crucial step to ensure a continuous timeline for the sliding window
all_replicates = []
for replicate_id in df['replicate_id'].unique():
    replicate_df = df[df['replicate_id'] == replicate_id]
    # Resample to daily frequency and forward-fill missing values
    replicate_df_resampled = replicate_df.asfreq('D').ffill()
    all_replicates.append(replicate_df_resampled)

df_processed = pd.concat(all_replicates)
print("Data resampled to daily frequency to handle missing dates.")

# --- Normalize the target variable ---
# LSTMs perform better with scaled data, typically in the  range
scaler = MinMaxScaler(feature_range=(0, 1))
df_processed['count_scaled'] = scaler.fit_transform(df_processed[['cumulative_mite_count']])
print("Cumulative mite count has been normalized.\n")

--- Step 1: Loading and Preprocessing Data ---
Dataset loaded and columns renamed successfully.


ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
# --- Part 2: Create Sequences using the Sliding Window Method ---
print("--- Step 2: Creating Training Sequences with Sliding Window ---")

def create_sequences(data, input_steps, output_steps):
    """
    Creates input (X) and output (y) sequences for a time-series model.
    """
    X, y = [], []
    for i in range(len(data) - input_steps - output_steps + 1):
        # The input is a sequence of 'input_steps' length
        X.append(data[i:(i + input_steps), 0])
        # The output is the following sequence of 'output_steps' length
        y.append(data[(i + input_steps):(i + input_steps + output_steps), 0])
    return np.array(X), np.array(y)

# Define the parameters for the sliding window
INPUT_STEPS = 30   # Use 30 days of history to predict
OUTPUT_STEPS = 15  # Predict the next 15 days

all_X, all_y = [], []

# Create sequences for each experimental replicate separately
for replicate_id in df_processed['replicate_id'].unique():
    replicate_data = df_processed[df_processed['replicate_id'] == replicate_id][['count_scaled']].values
    if len(replicate_data) > INPUT_STEPS + OUTPUT_STEPS:
        X_rep, y_rep = create_sequences(replicate_data, INPUT_STEPS, OUTPUT_STEPS)
        all_X.append(X_rep)
        all_y.append(y_rep)

# Combine sequences from all replicates into a single training dataset
X_train_ts = np.concatenate(all_X)
y_train_ts = np.concatenate(all_y)

# Reshape input to the 3D format required by LSTMs: [samples, timesteps, features]
X_train_ts = X_train_ts.reshape((X_train_ts.shape, X_train_ts.shape[1], 1))

print(f"Created {X_train_ts.shape} training sequences.")
print(f"Shape of input data (X): {X_train_ts.shape}")
print(f"Shape of output data (y): {y_train_ts.shape}\n")


# --- Part 3: Build and Train the Bi-LSTM Model ---
print("--- Step 3: Building and Training the Bi-LSTM Model ---")

# Define the model architecture
bi_lstm_model = Sequential()

# Compile the model
bi_lstm_model.compile(optimizer='adam', loss='mean_squared_error')
bi_lstm_model.summary()

# Train the model
print("\nTraining Bi-LSTM model...")
# We use a portion of the data for validation to monitor the training process
history = bi_lstm_model.fit(
    X_train_ts,
    y_train_ts,
    epochs=100, # Increase epochs for better learning, can be tuned
    batch_size=64,
    validation_split=0.2, # Use 20% of data for validation
    verbose=1
)
print("Bi-LSTM model training complete.\n")


# --- Part 4: Generate and Visualize a 15-Day Forecast ---
print("--- Step 4: Generating and Visualizing a Forecast ---")

# Select a specific replicate to forecast, e.g., 'T2R5'
test_replicate_id = 'T2R5'
print(f"Generating a 15-day forecast for replicate: {test_replicate_id}")

# Get the last INPUT_STEPS days of data for the chosen replicate to use as input
historical_data_scaled = df_processed[df_processed['replicate_id'] == test_replicate_id][['count_scaled']].tail(INPUT_STEPS).values
historical_data_reshaped = historical_data_scaled.reshape((1, INPUT_STEPS, 1))

# Use the trained model to predict the future
predicted_scaled = bi_lstm_model.predict(historical_data_reshaped)

# Inverse transform the prediction to get the actual count values
predicted_counts = scaler.inverse_transform(predicted_scaled).flatten()

# Prepare data for plotting
last_date = df_processed[df_processed['replicate_id'] == test_replicate_id].index[-1]
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=OUTPUT_STEPS)
historical_dates = df_processed[df_processed['replicate_id'] == test_replicate_id].index
historical_actual_counts = df_processed[df_processed['replicate_id'] == test_replicate_id]['cumulative_mite_count'].values

# Visualize the historical data and the forecast
plt.figure(figsize=(15, 7))
plt.plot(historical_dates, historical_actual_counts, label='Historical Actual Count', color='blue', linewidth=2)
plt.plot(forecast_dates, predicted_counts, label='15-Day Forecast', color='red', marker='o', linestyle='--')
plt.title(f'Spider Mite Count Forecast for Replicate {test_replicate_id}', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Mite Count', fontsize=12)
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

